# Application Example: Generic Election

Elsim is not limited to the Bundestag election. This notebook shows the complete workflow for any election — from configuring parties and constituencies to evaluating the result.

For each configuration, two approaches are available:
- **Option A (Interactive):** Widget UI — only available in a running Jupyter server.
- **Option B (Programmatic):** Direct configuration in code — runs anywhere.

For an explanation of the electoral procedure: [Introduction](../../docs/source/en/introduction.md)  
For a simulation with real election data: [federal_election.ipynb](federal_election.ipynb)

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

from ipres import (
    Election, ElectionConfig, Parties, ConstituenciesConfig,
    ElectionEvaluator,
)

---

## 1. Configure Parties

### Option A — Interactive

The widget UI allows loading parties from a file or generating them randomly.

In [ ]:
# Interactive only — widget appears below this cell when running in Jupyter
parties = Parties()
parties.setup()

### Option B — Programmatic

In [ ]:
parties = Parties.from_dataframe(pd.DataFrame({
    'party_name': [
        'Progress Party',
        "People's Union",
        'Green Future',
        'Liberal Centre',
        'Regional Party',
    ]
}))
display(parties.getParties())

---

## 2. Configure Constituencies

### Option A — Interactive

In [ ]:
# Interactive only — widget appears below this cell when running in Jupyter
cc = ConstituenciesConfig()
cc.setup()

### Option B — Programmatic

In [ ]:
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': [
        'Northtown', 'Southtown', 'East District', 'West District',
        'Midtown', 'Mountain Village', 'Lake City', 'Forest District',
    ],
    'constituency_size': [
        120_000, 95_000, 85_000, 110_000,
        75_000, 45_000, 130_000, 60_000,
    ],
    'turnout_percent': [
        72.0, 68.5, 74.2, 70.8,
        65.3, 71.1, 76.5, 69.4,
    ],
}))
display(cc.getConstituencies())

---

## 3. Election Configuration

`ElectionConfig` combines parties and constituencies and automatically computes parliament size and majority thresholds.

In [ ]:
party_names = parties.getPartyNames()
config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=party_names,
    seed=42,
)
print(f"Parties:          {party_names}")
print(f"Constituencies:   {cc.getM()}")
print(f"Parliament seats: {config.parliamentarySeats}")
print(f"Government seats: {config.getParliamentMajoritySeats()}")
print(f"Ballot majority:  {config.getBallotMajorityPercent():.1f} %")

---

## 4. Run the Election

`election.run()` runs all rounds automatically until a winner is determined. The optional `on_iteration_finished` callback allows tracking each round's result.

In [ ]:
rounds = []

def on_round_finished(iteration):
    rounds.append(iteration)
    n = len(rounds)
    if iteration.hasWinner():
        print(f"Round {n}: winner \u2192 {iteration.getWinner().name}")
    else:
        print(f"Round {n}: no winner")
    display(iteration.show_results_table(styler=True))

election = Election(electionConfig=config)
final = election.run(on_iteration_finished=on_round_finished)

print(f"\nTotal rounds: {election.getNumberOfIterations()}")
print(f"Election winner: {final.getWinner().name}")

---

## 5. Evaluate the Result

`ElectionEvaluator.evaluate()` runs the three evaluation steps automatically:
seat distribution → constituency count assignment → constituency assignment.

For details on each step and its configuration options: [`election_evaluation.ipynb`](election_evaluation.ipynb)

In [ ]:
evaluator = ElectionEvaluator(seed=42)
result = evaluator.evaluate(election)

In [ ]:
display(result.get_seat_distribution_table())

In [ ]:
display(result.get_constituency_summary_table())
display(result.get_constituency_assignment_table(sort_by='constituency'))

In [ ]:
fig = result.plot_seat_share_pie()
display(fig)
plt.close(fig)